# Pratica classe 2 · Regresión aplicada a un caso de uso nuevo

Nella lezione teorica construiste un modello de regresión lineal sobre una serie de precios simulada. Ora tocca a te a ti aplicar exactamente el mismo flujo de trabajo (features, label, `train_test_split`, addestramento, `.score()` y `shift` para forecasting) a un caso de uso distinto: el **consumo eléctrico simulado de una vivienda**.

Este notebook usa una serie simulada nueva, generada con NumPy y una semilla distinta a la de la lezione, para que practiques con dati que no has visto antes. El procedimiento es idéntico al del capítulo 2 del manual: solo cambia el contexto.

Encontrarás cellas con `# TODO: ...` que debes completar tú mismo o tú misma. Justo debajo de cada esercizio tienes una soluzione colapsada por si te atascas, pero te recomendamos intentarlo primero sin mirarla.

Empecemos generando los dati.

## Preparamos el escenario: consumo eléctrico simulado

Imagina que trabajas para una empresa de gestión energética y quieres anticipar el consumo eléctrico diario de una vivienda para planificar mejor la red. No tienes un sensor con años de histórico todavía, así que, para practicar el flujo de trabajo, simulas una serie con NumPy: un paseo aleatorio (random walk) alrededor de un consumo medio, con variaciones diarias.

Ejecuta la cella siguiente tal cual (esta parte ya está resuelta, porque el objetivo de la pratica es el modelado, no la simulación):

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(7)  # semilla distinta a la de la clase de teoría (allí se usó 42)

n_dias = 500
consumo_base = 15  # kWh medios por día
variacion_diaria = np.random.normal(loc=0.001, scale=0.03, size=n_dias)
consumo = consumo_base * (1 + variacion_diaria).cumprod()

fechas = pd.date_range(start="2023-01-01", periods=n_dias, freq="D")

df = pd.DataFrame({"Consumo": consumo}, index=fechas)
df.index.name = "Fecha"

# Temperatura simulada del día (afecta al consumo, más frío o más calor = más consumo)
df["Temperatura"] = 15 + 10 * np.sin(np.linspace(0, 6 * np.pi, n_dias)) + np.random.normal(0, 2, n_dias)

# Número de personas en la vivienda ese día (varía poco, entre 1 y 5)
df["Ocupantes"] = np.random.randint(1, 6, size=n_dias)

print(df.head())


## Ejercicio 1 · Construye las features y el label

**Enunciado**: tal como hiciste en lezione con `HL_percent` y `PCT_change`, crea una feature nueva llamada `Variacion_pct` que recoja el cambio porcentual del consumo respecto al día anterior. Después, define la lista `colonnas_features` con las colonnas `"Consumo"`, `"Temperatura"`, `"Ocupantes"` y `"Variacion_pct"`. Por último, crea el label desplazando la colonna `"Consumo"` `FORECAST_OUT` días hacia el futuro con `shift`.

**Pistas**:
- El cambio porcentual respecto al día anterior se calcula con `df["Consumo"].pct_change() * 100`
- Recuerda que `shift(-FORECAST_OUT)` desplaza la colonna hacia arriba, dejando `NaN` en las últimas rigas
- Define `FORECAST_OUT` como una variabile, igual que en lezione (por ejemplo, 14 días)

Completa el código en la cella siguiente:

In [ ]:
FORECAST_COL = "Consumo"
FORECAST_OUT = 14  # número de días que queremos predecir hacia el futuro

# TODO: crea la columna "Variacion_pct" con el cambio porcentual de "Consumo" respecto al día anterior


# TODO: define la lista columnas_features con "Consumo", "Temperatura", "Ocupantes" y "Variacion_pct"
columnas_features = None

# TODO: crea la columna "label" desplazando FORECAST_COL FORECAST_OUT días hacia el futuro (shift)


print(df[["Consumo", "label"]].head())
print(df[["Consumo", "label"]].tail())


<details>
<summary>💡 Vedi soluzione</summary>

```python
FORECAST_COL = "Consumo"
FORECAST_OUT = 14  # número de días que queremos predecir hacia el futuro

# Cambio porcentual del consumo respecto al día anterior
df["Variacion_pct"] = df["Consumo"].pct_change() * 100
df["Variacion_pct"] = df["Variacion_pct"].fillna(0)

# Lista de colonnas que usaremos como features
colonnas_features = ["Consumo", "Temperatura", "Ocupantes", "Variacion_pct"]

# Label: el consumo real dentro de FORECAST_OUT días
df["label"] = df[FORECAST_COL].shift(-FORECAST_OUT)

print(df[["Consumo", "label"]].head())
print(df[["Consumo", "label"]].tail())
```

</details>

## Ejercicio 2 · Separa addestramento y prueba, y entrena el modello

**Enunciado**: siguiendo el mismo procedimiento que en lezione, convierte las features en un array de NumPy y escálalas con `preprocessing.scale`. Guarda las rigas más recientes (las que no tienen label) en `X_lately`, elimina esas rigas de `X`, y quita del dataframe las rigas con `NaN` antes de construir `y`. Después, reparte los dati con `train_test_split` (80% addestramento, 20% prueba, `random_state=42`), crea un modello `LinearRegression` y entrénalo.

**Pistas**:
- El orden importa: primero separas `X_lately` con `X[-FORECAST_OUT:]`, luego recortas `X` con `X[:-FORECAST_OUT]`, y solo después haces `df.dropna(inplace=True)` para construir `y`
- `train_test_split(X, y, test_size=0.2, random_state=42)` devuelve cuatro elementos en este orden: `X_train, X_test, y_train, y_test`
- No olvides importar lo necesario de `sklearn`

Completa el código en la cella siguiente:

In [ ]:
from sklearn import preprocessing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# TODO: convierte columnas_features en un array de NumPy llamado X y escálalo con preprocessing.scale
X = None

# TODO: guarda en X_lately las últimas FORECAST_OUT filas de X, y recorta X para quitarlas


# TODO: elimina del dataframe las filas con NaN (df.dropna) y construye y a partir de df["label"]


# TODO: separa X e y en entrenamiento y prueba con train_test_split (test_size=0.2, random_state=42)


# TODO: crea el modelo LinearRegression y entrénalo con fit()
modelo = None

# TODO: calcula la puntuación del modelo sobre el conjunto de prueba con .score() y guárdala en "precision"
precision = None

print("Precisión del modelo (R²):", round(precision, 4))


<details>
<summary>💡 Vedi soluzione</summary>

```python
from sklearn import preprocessing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Features como array de NumPy, escaladas
X = np.array(df[colonnas_features])
X = preprocessing.scale(X)

# Filas más recientes, sin label conocido, reservadas para el forecasting
X_lately = X[-FORECAST_OUT:]
X = X[:-FORECAST_OUT]

# Quitamos las rigas sin label antes de construir y
df.dropna(inplace=True)
y = np.array(df["label"])

# Separación en addestramento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Entrenamiento del modello
modello = LinearRegression(n_jobs=-1)
modello.fit(X_train, y_train)

# Evaluación sobre dati que el modello no vio en el addestramento
precision = modello.score(X_test, y_test)
print("Precisión del modello (R²):", round(precision, 4))
```

</details>

## Ejercicio 3 · Predice el consumo futuro y represéntalo con Matplotlib

**Enunciado**: usa el modello ya entrenado para predecir el consumo de los próximos `FORECAST_OUT` días a partir de `X_lately`. Después, añade una colonna `"Forecast"` al dataframe, rellénala riga a riga con fechas futuras (igual que en lezione) y dibuja un gráfico con el histórico y la predicción usando Matplotlib.

**Pistas**:
- `modello.predict(X_lately)` te devuelve un valore por cada una de las rigas futuras
- Puedes reutilizar el mismo bucle con `datetime.timedelta(days=1)` que viste en la lezione de teoría
- No olvides poner `plt.legend()`, `plt.xlabel()`, `plt.ylabel()` y `plt.title()` para que el gráfico se entienda de un vistazo

Completa el código en la cella siguiente:

In [ ]:
import datetime
import matplotlib.pyplot as plt

# TODO: genera las predicciones para X_lately con el modelo entrenado
predicciones = None

# TODO: crea la columna "Forecast" vacía (np.nan) en el dataframe


# TODO: recorre "predicciones" y añade cada una en una fila nueva con una fecha posterior a la última,
# tal como hiciste en la clase de teoría


# TODO: dibuja df["Consumo"] y df["Forecast"] con Matplotlib, con leyenda, etiquetas de ejes y título



<details>
<summary>💡 Vedi soluzione</summary>

```python
import datetime
import matplotlib.pyplot as plt

# Predicciones para los próximos FORECAST_OUT días
predicciones = modello.predict(X_lately)

# Columna vacía donde iremos guardando las predicciones futuras
df["Forecast"] = np.nan

ultima_fecha = df.index[-1]
un_dia = datetime.timedelta(days=1)
siguiente_fecha = ultima_fecha + un_dia

for prediccion in predicciones:
    riga_vacia = [np.nan for _ in range(len(df.columns) - 1)]
    df.loc[siguiente_fecha] = riga_vacia + [prediccion]
    siguiente_fecha += un_dia

df["Consumo"].plot(label="Consumo histórico simulado")
df["Forecast"].plot(label=f"Predicción a {FORECAST_OUT} días")
plt.legend(loc="best")
plt.xlabel("Fecha")
plt.ylabel("Consumo eléctrico simulado (kWh)")
plt.title("Consumo eléctrico simulado de una vivienda y su predicción")
plt.show()
```

</details>

## Caso de uso adicional (opcional): demanda simulada de un producto

Si te ha sobrado tiempo o quieres reforzar lo aprendido, repite tú mismo o tú misma todo el flujo (Ejercicios 1 a 3) con una serie distinta: la **demanda diaria simulada de un producto** en un almacén.

Genera la serie con este código (ya resuelto) y luego copia y adapta tus soluciones de los esercizios anteriores en las cellas de código en blanco que siguen:

In [ ]:
np.random.seed(23)  # otra semilla distinta, para tener una serie nueva

n_dias_demanda = 500
demanda_base = 200  # unidades medias vendidas por día
variacion_demanda = np.random.normal(loc=0.0, scale=0.04, size=n_dias_demanda)
demanda = demanda_base * (1 + variacion_demanda).cumprod()

fechas_demanda = pd.date_range(start="2023-06-01", periods=n_dias_demanda, freq="D")

df_demanda = pd.DataFrame({"Demanda": demanda}, index=fechas_demanda)
df_demanda.index.name = "Fecha"

# Precio de venta simulado (a más precio, en general, algo menos de demanda)
df_demanda["Precio"] = 20 + np.random.normal(0, 1.5, n_dias_demanda)

# Si el día fue promoción o no (0 = no, 1 = sí)
df_demanda["Promocion"] = np.random.randint(0, 2, size=n_dias_demanda)

print(df_demanda.head())


**Enunciado del reto**: aplica sobre `df_demanda` los mismos tres pasos que sobre `df`:

- Crea una feature de variación porcentual de `"Demanda"`, define `colonnas_features_demanda` con `"Demanda"`, `"Precio"`, `"Promocion"` y tu nueva feature, y crea el `"label"` con `shift`
- Separa `X`/`y`, escala, reserva `X_lately`, haz el `train_test_split`, entrena un `LinearRegression` y calcula el `.score()`
- Predice con `.predict()` y dibuja el histórico junto con la predicción

Usa la cella de código en blanco de abajo. No hay soluzione colapsada para este reto adicional: es tu oportunidad de aplicar todo el flujo da solo, de principio a fin.

In [ ]:
# TODO (reto adicional): reproduce aquí, sobre df_demanda, el flujo completo de los ejercicios 1 a 3
# (features y label, train_test_split, entrenamiento, score y forecasting con gráfico)



## Per concludere la pratica

Repasa lo que acabas de hacer:

- Simulaste dos series de dati nuevas con NumPy, cada una con su propia semilla
- Construiste features con significado (variación porcentual, temperatura, ocupantes, precio, promoción)
- Aplicaste `shift` para convertir un valore futuro en el label que quieres predecir
- Entrenaste y evaluaste un modello `LinearRegression` con `train_test_split` y `.score()`
- Usaste `.predict()` sobre las rigas sin label para generar un forecasting y lo representaste con Matplotlib

¿Ves cómo el procedimiento es exactamente el mismo que en la lezione de teoría, aunque el contexto (consumo eléctrico o demanda de un producto) sea completamente distinto? Esa es la idea central de esta lección: el flujo de trabajo de scikit-learn se repite, cambies lo que cambies como dato de partida.

**Abbreviazioni usate en esta pratica:** kWh (kilovatio-hora).